In [0]:
#-------------------------------------------------------------------
# Crear logger visible para todos los modulos
#-------------------------------------------------------------------
import logging

logger = logging.getLogger("Muric")

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.setLevel(logging.INFO)

logger.info("Logger creado correctamente.")

In [0]:
import os

secret_scope = os.getenv("JAVA_HOME")
if not secret_scope:
    raise Exception("No se ha encontrado la variable de entorno")

SEVERITY_MAP={
    "VERBOSE": 0,
    "INFO":    1,
    "WARNING": 2,
    "ERROR":   3,
    "CRITICAL":4
}

parts=secret_scope.split("/")

ruta="abfss://zona-cruda@stsfcdevlakehouse.dfs.core.windows.net/ArchivosMuricp7z/53638257_T32_C5_muric_30062025.avro"

#remover prefijo
clean_path=ruta.replace("abfss://","")
container=clean_path.split("@")[0]
print (f'Contenedor: {container}')
storage_acount=clean_path.split("@")[1].split(".")[0]
print (f'Cuenta de almacenamiento: {storage_acount}')
internal_path=clean_path.split(".net/")[1]
print (f'Path interno: {internal_path}')
file_name_raw=internal_path.split("/")[-1]
print(f'Archivo: {file_name_raw}')
folder=internal_path.split("/")[0]
print(f'Carpeta: {folder}')




In [0]:
#/Volumes/workspace/default/misarchivos
#--------------------------------------
#Lectura de un archivo JSON
#--------------------------------------

from pyspark.sql.functions import explode
import json

ruta="""/Volumes/workspace/default/misarchivos/prueba.json"""

df = spark.read.option("multiline", "true").json(ruta)
#df.printSchema()
#df.show(truncate=False)
df_exploded=df.withColumn('item',explode("items"))
df_final = df_exploded.select(
    "pedido_id",
    "cliente",
    "item.producto",
    "item.precio"
)
df_final.show()

df_final.write.format("delta").mode('overwrite').saveAsTable('PedidosBronze2')
spark.sql('SELECT * FROM PedidosBronze2 WHERE precio>50;').show()
spark.sql('DROP TABLE IF EXISTS PedidosBronze')